# Task 0.5 — Complete AlphaMissense annotations

Retrieve missing AlphaMissense scores for nonsynonymous variants from the AlphaFold DB per-protein annotation files. Preserve source-workbook scores, record provenance and query status, and leave synonymous substitutions unscored.

The canonical `KEY` is `Gene||Mutation_used`. This notebook does not impute unavailable scores.

In [1]:
from pathlib import Path
import json
import re
import time

import numpy as np
import pandas as pd
import requests

BASE = Path("/mnt/volume6/czj/labLGN/LabLZ")
INPUT_CSV = BASE / "data_preparation/cell2024_model_single_subst.csv"
OUTPUT_CSV = BASE / "xgboost_trial/alphamissense_completed.csv"
CACHE_DIR = BASE / "xgboost_trial/alphamissense_cache"
AFDB_API = "https://alphafold.ebi.ac.uk/api/prediction/{accession}"
REQUEST_TIMEOUT = 60
MAX_RETRIES = 3
REQUEST_DELAY = 0.2

CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Input:  {INPUT_CSV}")
print(f"Output: {OUTPUT_CSV}")
print(f"Cache:  {CACHE_DIR}")

Input:  /mnt/volume6/czj/labLGN/LabLZ/data_preparation/cell2024_model_single_subst.csv
Output: /mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/alphamissense_completed.csv
Cache:  /mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/alphamissense_cache


In [2]:
df = pd.read_csv(INPUT_CSV)
df["KEY"] = (df["Gene"].astype(str).str.strip() + "||"
                    + df["Mutation_used"].astype(str).str.strip())

mutation_parts = df["Mutation_used"].str.extract(r"^([A-Z])(\d+)([A-Z])$")
mutation_parts.columns = ["wt_aa", "position", "mutant_aa"]
df = pd.concat([df, mutation_parts], axis=1)
df["position"] = pd.to_numeric(df["position"], errors="raise").astype(int)
df["is_synonymous"] = df["wt_aa"].eq(df["mutant_aa"])

assert len(df) == 2179
assert df["KEY"].is_unique
assert mutation_parts.notna().all(axis=None), "Unparseable Mutation_used value"
assert df["Mislocalized"].isin([0, 1]).all()
assert int(df["Mislocalized"].sum()) == 236

missing_original = df["AlphaMissense score"].isna()
query_mask = missing_original & ~df["is_synonymous"]
print(f"Rows: {len(df)}")
print(f"Original AlphaMissense scores: {(~missing_original).sum()}")
print(f"Missing nonsynonymous variants to query: {query_mask.sum()}")
print(f"Missing synonymous variants (not applicable): {(missing_original & df['is_synonymous']).sum()}")
assert int(query_mask.sum()) == 95
assert int((missing_original & df["is_synonymous"]).sum()) == 31

Rows: 2179
Original AlphaMissense scores: 2053
Missing nonsynonymous variants to query: 95
Missing synonymous variants (not applicable): 31


In [3]:
def request_json(session, url):
    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            response = session.get(url, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError) as exc:
            last_error = exc
            if attempt + 1 < MAX_RETRIES:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Request failed after {MAX_RETRIES} attempts: {url}") from last_error

def download_file(session, url, output_path):
    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            response = session.get(url, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            output_path.write_bytes(response.content)
            return
        except requests.RequestException as exc:
            last_error = exc
            if attempt + 1 < MAX_RETRIES:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Download failed after {MAX_RETRIES} attempts: {url}") from last_error

def find_annotation_url(metadata):
    records = metadata if isinstance(metadata, list) else [metadata]
    preferred_keys = ["amAnnotationsUrl", "am_annotations_url"]
    for record in records:
        for key in preferred_keys:
            if record.get(key):
                return record[key], record
    return None, records[0] if records else {}

def standardise_am_table(am_df):
    aliases = {
        "protein_variant": ["protein_variant", "proteinVariant", "variant"],
        "am_pathogenicity": ["am_pathogenicity", "amPathogenicity", "pathogenicity"],
        "am_class": ["am_class", "amClass", "classification"],
    }
    rename = {}
    for canonical, candidates in aliases.items():
        found = next((name for name in candidates if name in am_df.columns), None)
        if found is None:
            raise ValueError(f"Missing {canonical}; columns={am_df.columns.tolist()}")
        rename[found] = canonical
    out = am_df.rename(columns=rename)[list(aliases)].copy()
    out["protein_variant"] = out["protein_variant"].astype(str).str.strip()
    out["am_pathogenicity"] = pd.to_numeric(out["am_pathogenicity"], errors="coerce")
    return out

def load_protein_am_table(accession, session):
    table_path = CACHE_DIR / f"{accession}_am_annotations.csv"
    metadata_path = CACHE_DIR / f"{accession}_metadata.json"

    if table_path.exists():
        return standardise_am_table(pd.read_csv(table_path)), "cache", None

    metadata = request_json(session, AFDB_API.format(accession=accession))
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    annotation_url, record = find_annotation_url(metadata)
    if annotation_url is None:
        return None, "no_annotation_url", record

    download_file(session, annotation_url, table_path)
    return standardise_am_table(pd.read_csv(table_path)), "downloaded", record

In [4]:
results = []
protein_tables = {}
session = requests.Session()
session.headers.update({"User-Agent": "MISFIT-research/1.0"})

query_df = df.loc[query_mask, ["KEY", "Uniprot", "Mutation_used"]].copy()
accessions = query_df["Uniprot"].drop_duplicates().tolist()
print(f"Downloading AlphaMissense tables for {len(accessions)} proteins")

for index, accession in enumerate(accessions, start=1):
    try:
        am_table, table_status, metadata = load_protein_am_table(accession, session)
        protein_tables[accession] = (am_table, table_status)
    except Exception as exc:
        protein_tables[accession] = (None, f"query_error:{type(exc).__name__}:{str(exc)[:120]}")
    if index % 10 == 0 or index == len(accessions):
        print(f"  proteins processed: {index}/{len(accessions)}")
    time.sleep(REQUEST_DELAY)

for row in query_df.itertuples(index=False):
    am_table, table_status = protein_tables[row.Uniprot]
    if am_table is None:
        results.append({
            "KEY": row.KEY,
            "retrieved_alphamissense_score": np.nan,
            "retrieved_alphamissense_class": None,
            "retrieval_status": table_status,
        })
        continue

    match = am_table[am_table["protein_variant"].eq(row.Mutation_used)]
    if len(match) == 1:
        hit = match.iloc[0]
        results.append({
            "KEY": row.KEY,
            "retrieved_alphamissense_score": hit["am_pathogenicity"],
            "retrieved_alphamissense_class": hit["am_class"],
            "retrieval_status": f"retrieved_{table_status}",
        })
    elif len(match) == 0:
        results.append({
            "KEY": row.KEY,
            "retrieved_alphamissense_score": np.nan,
            "retrieved_alphamissense_class": None,
            "retrieval_status": "variant_not_found",
        })
    else:
        raise ValueError(f"Duplicate AlphaMissense entries for {row.KEY}")

retrieved = pd.DataFrame(results)
assert retrieved["KEY"].is_unique
print(retrieved["retrieval_status"].value_counts(dropna=False))

  proteins processed: 10/37
  proteins processed: 20/37
  proteins processed: 30/37
  proteins processed: 37/37
retrieval_status
retrieved_cache                                                                                                87
no_annotation_url                                                                                               7
query_error:RuntimeError:Request failed after 3 attempts: https://alphafold.ebi.ac.uk/api/prediction/P07203     1
Name: count, dtype: int64


In [5]:
output_cols = [
    "KEY", "Gene", "Uniprot", "Mutation_used",
    "wt_aa", "position", "mutant_aa", "is_synonymous",
    "Mislocalized", "source",
]
out = df[output_cols].copy()
out["original_alphamissense_score"] = df["AlphaMissense score"]
out["original_alphamissense_class"] = df["AlphaMissense class"]
out = out.merge(retrieved, on="KEY", how="left", validate="one_to_one")

out["final_alphamissense_score"] = out["original_alphamissense_score"].combine_first(
    out["retrieved_alphamissense_score"]
)
out["final_alphamissense_class"] = out["original_alphamissense_class"].combine_first(
    out["retrieved_alphamissense_class"]
)

out["alphamissense_status"] = "unresolved"
out.loc[out["original_alphamissense_score"].notna(), "alphamissense_status"] = "original"
out.loc[out["original_alphamissense_score"].isna() & out["is_synonymous"],
        "alphamissense_status"] = "not_applicable_synonymous"
retrieved_ok = (out["original_alphamissense_score"].isna()
                & out["retrieved_alphamissense_score"].notna())
out.loc[retrieved_ok, "alphamissense_status"] = "retrieved"
unresolved = (out["original_alphamissense_score"].isna()
              & ~out["is_synonymous"]
              & out["retrieved_alphamissense_score"].isna())
out.loc[unresolved, "alphamissense_status"] = out.loc[unresolved, "retrieval_status"].fillna("unresolved")
out["alphamissense_source"] = np.select(
    [out["alphamissense_status"].eq("original"), out["alphamissense_status"].eq("retrieved")],
    ["source_workbook", "AlphaFold_DB_amAnnotationsUrl"],
    default=None,
)

assert len(out) == 2179
assert out["KEY"].is_unique
assert out["final_alphamissense_score"].dropna().between(0, 1).all()
assert out.loc[out["alphamissense_status"].eq("not_applicable_synonymous"),
               "final_alphamissense_score"].isna().all()

out.to_csv(OUTPUT_CSV, index=False)
print(out["alphamissense_status"].value_counts(dropna=False))
print(f"Final available scores: {out['final_alphamissense_score'].notna().sum()}/{len(out)}")
print(f"Saved: {OUTPUT_CSV}")

alphamissense_status
original                                                                                                       2053
retrieved                                                                                                        87
not_applicable_synonymous                                                                                        31
no_annotation_url                                                                                                 7
query_error:RuntimeError:Request failed after 3 attempts: https://alphafold.ebi.ac.uk/api/prediction/P07203       1
Name: count, dtype: int64
Final available scores: 2140/2179
Saved: /mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/alphamissense_completed.csv
